# 6.2 - Khai phá tập phổ biến và luật kết hợp với thư viện PyFIM (Apriori)
**Họ và tên:** Thái Anh Lạc | **MSSV:** 089206000860 | **Mã lớp HP:** 012012204103

In [1]:
# 1. Nạp thư viện FIM và dữ liệu Market-Basket
from fim import apriori, arules

dataset = [
    ['Bread', 'Milk'],
    ['Bread', 'Diaper', 'Beer', 'Eggs'],
    ['Milk', 'Diaper', 'Beer', 'Coke'],
    ['Bread', 'Milk', 'Diaper', 'Beer'],
    ['Bread', 'Milk', 'Diaper', 'Coke']
]

N = len(dataset)
min_sup = 40   # minSup = 40% (tối thiểu 2/5 giao dịch)
min_conf = 60  # minConf = 60%

print(f"Số lượng giao dịch: {N}")
print(f"Cấu hình: minSup = {min_sup}% (tối thiểu {int(min_sup*N/100)}/{N} giao dịch), minConf = {min_conf}%")

Số lượng giao dịch: 5
Cấu hình: minSup = 40% (tối thiểu 2/5 giao dịch), minConf = 60%


In [2]:
# 2. Khai phá các tập phổ biến (Frequent Itemsets) bằng fim.apriori
# - supp=40: độ hỗ trợ tối thiểu 40%
# - zmin=1: kích thước tập mục tối thiểu từ 1 phần tử
# - report='as': xuất số lượng giao dịch tuyệt đối (a) và tỷ lệ hỗ trợ (s)
frequent_itemsets = apriori(dataset, supp=min_sup, zmin=1, report='as')

# Phân loại và in theo từng cấp độ k
by_k = {}
for itemset, sup_cnt, sup_frac in frequent_itemsets:
    k = len(itemset)
    by_k.setdefault(k, []).append((itemset, sup_cnt, sup_frac))

print(f"=== KẾT QUẢ TẬP PHỔ BIẾN BẰNG FIM.APRIORI (Tổng: {len(frequent_itemsets)} tập) ===")
for k in sorted(by_k.keys()):
    print(f"\nFrequent {k}-itemset ({len(by_k[k])} tập):")
    for itemset, sup_cnt, sup_frac in sorted(by_k[k], key=lambda x: x[1], reverse=True):
        print(f"  {set(itemset)} | Support = {sup_frac:.2f} ({sup_cnt}/{N}) | {sup_frac*100:.0f}%")
print("=" * 65)

=== KẾT QUẢ TẬP PHỔ BIẾN BẰNG FIM.APRIORI (Tổng: 17 tập) ===

Frequent 1-itemset (5 tập):
  {'Bread'} | Support = 0.80 (4/5) | 80%
  {'Milk'} | Support = 0.80 (4/5) | 80%
  {'Diaper'} | Support = 0.80 (4/5) | 80%
  {'Beer'} | Support = 0.60 (3/5) | 60%
  {'Coke'} | Support = 0.40 (2/5) | 40%

Frequent 2-itemset (8 tập):
  {'Diaper', 'Beer'} | Support = 0.60 (3/5) | 60%
  {'Bread', 'Milk'} | Support = 0.60 (3/5) | 60%
  {'Diaper', 'Bread'} | Support = 0.60 (3/5) | 60%
  {'Diaper', 'Milk'} | Support = 0.60 (3/5) | 60%
  {'Milk', 'Coke'} | Support = 0.40 (2/5) | 40%
  {'Diaper', 'Coke'} | Support = 0.40 (2/5) | 40%
  {'Bread', 'Beer'} | Support = 0.40 (2/5) | 40%
  {'Milk', 'Beer'} | Support = 0.40 (2/5) | 40%

Frequent 3-itemset (4 tập):
  {'Diaper', 'Milk', 'Coke'} | Support = 0.40 (2/5) | 40%
  {'Diaper', 'Bread', 'Beer'} | Support = 0.40 (2/5) | 40%
  {'Diaper', 'Milk', 'Beer'} | Support = 0.40 (2/5) | 40%
  {'Diaper', 'Bread', 'Milk'} | Support = 0.40 (2/5) | 40%


In [3]:
# 3. Sinh các luật kết hợp (Association Rules) bằng fim.arules
# - supp=40: support tối thiểu 40%
# - conf=60: confidence tối thiểu 60%
# - zmin=2: tối thiểu 2 mục (tiền đề có ít nhất 1 mục)
# - report='asCl': xuất absolute support (a), support fraction (s), confidence % (C), lift (l)
rules = arules(dataset, supp=min_sup, conf=min_conf, zmin=2, report='asCl')

# Sắp xếp các luật theo Confidence giảm dần, sau đó theo Lift và Support
sorted_rules = sorted(rules, key=lambda x: (x[4], x[5], x[3]), reverse=True)

print(f"DANH SÁCH CÁC LUẬT KẾT HỢP (minConf = {min_conf}%, minSup = {min_sup}%) - Tổng: {len(sorted_rules)} luật:\n")
for idx, (head, body, sup_cnt, sup_frac, conf_pct, lift_val) in enumerate(sorted_rules, 1):
    ant_str = str(set(body))
    con_str = str(set([head]))
    print(f"{idx:2d}. {ant_str:<25} => {con_str:<12} | Support = {sup_frac:.2f} ({sup_cnt}/{N}) | Conf = {conf_pct:5.1f}% | Lift = {lift_val:.3f}")

DANH SÁCH CÁC LUẬT KẾT HỢP (minConf = 60%, minSup = 40%) - Tổng: 24 luật:

 1. {'Beer'}                  => {'Diaper'}   | Support = 0.60 (3/5) | Conf = 100.0% | Lift = 1.250
 2. {'Milk', 'Beer'}          => {'Diaper'}   | Support = 0.40 (2/5) | Conf = 100.0% | Lift = 1.250
 3. {'Bread', 'Beer'}         => {'Diaper'}   | Support = 0.40 (2/5) | Conf = 100.0% | Lift = 1.250
 4. {'Coke'}                  => {'Diaper'}   | Support = 0.40 (2/5) | Conf = 100.0% | Lift = 1.250
 5. {'Milk', 'Coke'}          => {'Diaper'}   | Support = 0.40 (2/5) | Conf = 100.0% | Lift = 1.250
 6. {'Diaper', 'Coke'}        => {'Milk'}     | Support = 0.40 (2/5) | Conf = 100.0% | Lift = 1.250
 7. {'Coke'}                  => {'Milk'}     | Support = 0.40 (2/5) | Conf = 100.0% | Lift = 1.250
 8. {'Diaper'}                => {'Beer'}     | Support = 0.60 (3/5) | Conf =  75.0% | Lift = 1.250
 9. {'Milk'}                  => {'Diaper'}   | Support = 0.60 (3/5) | Conf =  75.0% | Lift = 0.938
10. {'Diaper'}           